# Ping Envelope fall

One continuous one-pole chase toward silence. No attack, no hold, no multipoint ADSR.

```text
env[0] = trigger_height
x      = feedback * env + offset
shape  = 0 if x <= 0 else 1 if x >= 1 else 10 ** (5 * (x - 1))
f      = shape * max_release_hz
k      = 1 - exp(-2 * pi * f / sample_rate)
env    = env + k * (0 - env)
```

`k` is 0 at `f <= 0` and 1 at or above Nyquist. The 5-decade shape is the sandbox Ping Envelope release map, shared by every curve here. **Short** and **Long** are presets of the four parameters, not a separate axis.

| | Feedback | Max release Hz | Offset | Trigger height |
| --- | --- | --- | --- | --- |
| Short (decay 0.5) | 1 | 10 | 0 | 1 |
| Long (decay 0.5) | 0.7718 | 1000 | 0.5 | 1 |
| Long (offset 0) | 0.7718 | 1000 | 0 | 1 |

Offset absorbs the module Decay knob: Short uses `0.5 - decay`, Long uses `1 - decay`.

The loop stops at amplitude **0.001** (−60 dB). **0.01** (−40 dB) is only a dashed marker. Sample rate is 48 kHz. The interactive sliders and overlay checkboxes are in `index.html` (run `python -m graphs.ping_envelope` from the repo root).

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for cand in [here, *here.parents]:
    if (cand / "graphs" / "ping_envelope" / "simulate.py").exists():
        sys.path.insert(0, str(cand))
        break
else:
    raise SystemExit("Open this notebook from the soemdsp-research repo.")

from graphs.ping_envelope.simulate import load_config, models, simulate_fall

config = load_config()
print(f"sample rate {config['sample_rate']} Hz, eng floor {config['eng_floor']}")
for model in models():
    curve = simulate_fall(model.params)
    print(
        f"{model.label:18} stop={curve.stopped:10} "
        f"-40dB={curve.t_perceptual_s}  -60dB={curve.t_eng_s}  "
        f"end={curve.end_amplitude:.6f}"
    )

In [ ]:
from graphs.ping_envelope.plot import build_figure

named = []
for model in models():
    named.append((model.label, model.color, simulate_fall(model.params)))

fig = build_figure(named, log_y=True, title="Ping Envelope fall — model presets")
fig.show()